In [2]:
!pip install -q langchain_experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 7.3 MB/s eta 0:00:00


In [3]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import OpenAIEmbeddings

**1. Better Chunking:**

In [4]:
text = """
Machine learning helps computers learn from data.
Supervised learning uses labeled examples.
Unsupervised learning finds hidden patterns.
Deep learning uses neural networks.
RAG combines retrieval and generation.
"""

In [ ]:
embeddings = OpenAIEmbeddings(api_key="API_KEY")

text_splitter = SemanticChunker(embeddings)
chunks = text_splitter.create_documents([text])

**2. Re-ranking:**

In [12]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [ ]:
compressor = LLMChainExtractor.from_llm(llm)

compressor_retriver = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retirver
)

**3. Hybrid Search:**

In [14]:
from langchain_classic.retrievers import BM25Retriever

In [ ]:
bm25_retriver = BM25Retriever.from_documents(chunks)
bm25_retriver.k = 3

In [ ]:
semantic_retriver = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
def hybrid_search(query):
    keyword_results = bm25_retriever.get_relevant_documents(query)
    semantic_results = semantic_retriever.get_relevant_documents(query)
    
    seen = set()
    combined_results = []
    
    for doc in keyword_results + semantic_results:
        if doc.page_content not in seen:
            seen.add(doc.page_content)
            combined_results.append(doc)
            
    return combined_results

In [ ]:
def hybrid_search2(query, k=5, alpha=0.5):
    keyword_results = bm25_retriever.get_relevant_documents(query)
    semantic_results = semantic_retriever.get_relevant_documents(query)

    doc_scores = {}
    doc_objects = {}

    # BM25 results scoring
    for rank, doc in enumerate(keyword_results):
        content = doc.page_content
        score = 1 / (rank + 1)
        if content not in doc_scores:
            doc_scores[content] = 0
            doc_objects[content] = doc
        doc_scores[content] += (1 - alpha) * score

    # Semantic results scoring
    for rank, doc in enumerate(semantic_results):
        content = doc.page_content
        score = 1 / (rank + 1)
        if content not in doc_scores:
            doc_scores[content] = 0
            doc_objects[content] = doc
        doc_scores[content] += alpha * score

    # Sort by combined score
    ranked_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)

    combined_results = [doc_objects[content] for content, _ in ranked_docs[:k]]

    return combined_results